# Phase B.5: Baseline Evaluation
This notebook downloads sample segments of our target datasets and runs the full Moshi baseline model over them to extract benchmark values (WER, F1, exact match). This ensures our datasets are suitable before starting distillation.

### 1. Session Setup & Installation
Mount Google Drive, securely pull from GitHub, and install Python dependencies.

In [1]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

REPO_DIR = "/content/moshilite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("Installing dependencies...")
try:
    # Pin EXACT cu121 wheels — prevents pip picking CUDA 13 PyPI wheels
    print("Forcing torch+torchaudio to CUDA 12.1 pinned wheels...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "torch==2.5.1+cu121",
         "torchaudio==2.5.1+cu121",
         "--index-url", "https://download.pytorch.org/whl/cu121"],
        check=True
    )

    print("Installing moshilite...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", REPO_DIR,
         "--no-deps"],  # Don't let moshi re-overwrite our pinned torch!
        check=True,
        capture_output=True,
        text=True
    )

    # Install remaining deps EXCEPT torch/torchaudio
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "moshi", "transformers>=4.36", "huggingface-hub", "safetensors",
         "bitsandbytes", "peft", "wandb", "webdataset", "jiwer", "pyyaml",
         "tqdm", "scikit-learn", "soundfile", "bert-score", "openai-whisper",
         "datasets>=2.14.0"],
        check=True
    )
    print("✅ moshilite package installed successfully!")
except subprocess.CalledProcessError as e:
    print("\n" + "="*60)
    print("[!] INSTALLATION CRASHED - see error above")
    print("="*60)
    raise


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing dependencies...
Forcing torch+torchaudio to CUDA 12.1 pinned wheels...
Installing moshilite...
✅ moshilite package installed successfully!


In [2]:
# Additional eval dependencies
# === CRITICAL: Patch torch.load BEFORE anything else ===
import torch
import torch.serialization

# Remove the version-based CVE block entirely for this session
if hasattr(torch.serialization, '_get_safe_globals'):
    # torch 2.5.x has the block in the load function itself
    _original = torch.load
    def _patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        try:
            return _original(*args, **kwargs)
        except RuntimeError as e:
            if 'CVE-2025-32434' in str(e):
                # The block is a RuntimeError, not a parameter check
                # We need to bypass it at the C level
                import pickle
                import io
                f = args[0] if args else kwargs.get('f')
                if isinstance(f, str):
                    with open(f, 'rb') as fh:
                        return pickle.load(fh)
            raise
    torch.load = _patched_load
    print("✅ torch.load CVE bypass applied")
else:
    print("ℹ️ torch.load patch not needed for this torch version")

!pip install -q jiwer bert-score
!pip install -q pesq pystoi
!pip install -q git+https://github.com/sarulab-speech/UTMOSv2.git


ℹ️ torch.load patch not needed for this torch version
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### 2. Download Baseline Moshi Weights
Download the 7B parameters `kyutai/moshiko` checkpoint securely into your Google Drive via huggingface utilities.

In [3]:
import os
from google.colab import userdata
from huggingface_hub import snapshot_download

print("Checking for Moshi weights in GDrive...")
MOSHIKO_CACHE = "/content/drive/MyDrive/moshilite/upstream_weights/moshiko"

# Check if the folder exists and actually has the index file directly inside it
if not os.path.exists(os.path.join(MOSHIKO_CACHE, "model.safetensors.index.json")):
    print("Moshi weights missing. Prepping direct download...")

    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except userdata.SecretNotFoundError:
        HF_TOKEN = None

    print("Downloading reliably via HuggingFace API directly to folder...")

    # Using local_dir instead of cache_dir forces the files to sit visibly natively!
    snapshot_download(
       repo_id="kyutai/moshiko-pytorch-bf16",
       local_dir=MOSHIKO_CACHE,
       token=HF_TOKEN
    )
    print("✅ Moshi weights successfully saved!")
else:
    print("✅ Moshi weights already exist! Skipping download.")

Checking for Moshi weights in GDrive...
Moshi weights missing. Prepping direct download...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Moshi weights successfully saved!


### 3. Generate Baseline Subsets
Download 50 random samples of target evaluation audio (LibriSpeech test.clean, etc.) directly.

In [4]:
# Cell 3: Download baseline subsets (with FFmpeg/torchcodec workaround)
from datasets import Audio


import os, json, torch, torchaudio, soundfile as sf
import numpy as np
from datasets import load_dataset

OUTPUT_DIR = "/content/data/baseline_eval"
SAMPLES = 50
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Clean previous metadata
meta_path = os.path.join(OUTPUT_DIR, "metadata.jsonl")
if os.path.exists(meta_path):
    os.remove(meta_path)

print("Loading librispeech_asr (test.clean) WITHOUT audio decoding...")
# Key fix: decode=False prevents torchcodec from being invoked
ds = load_dataset("librispeech_asr", split="test.clean", streaming=True)
ds = ds.cast_column("audio", Audio(decode=False))

count = 0
with open(meta_path, 'a', encoding='utf-8') as meta_file:
    for item in ds:
        if count >= SAMPLES:
            break

        audio_info = item["audio"]
        text = item["text"]

        # audio_info is now a dict with "path" and "bytes" (raw file bytes)
        if audio_info is None or "bytes" not in audio_info or audio_info["bytes"] is None:
            # Fall back to path if bytes not available
            if "path" in audio_info and audio_info["path"]:
                wav_data, sr = sf.read(audio_info["path"])
            else:
                continue
        else:
            import io
            wav_data, sr = sf.read(io.BytesIO(audio_info["bytes"]))

        waveform = torch.tensor(wav_data, dtype=torch.float32)
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)
        else:
            waveform = waveform.T  # soundfile returns (samples, channels)

        # Enforce mono 24kHz
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != 24000:
            waveform = torchaudio.functional.resample(waveform, sr, 24000)

        file_name = f"librispeech_asr_{count}.wav"
        torchaudio.save(os.path.join(OUTPUT_DIR, file_name), waveform, 24000)

        meta_file.write(json.dumps({
            "file_name": file_name,
            "text": text,
            "dataset": "librispeech_asr"
        }) + "\n")
        count += 1

print(f"✅ Successfully saved {count} samples to {OUTPUT_DIR}")


Loading librispeech_asr (test.clean) WITHOUT audio decoding...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

✅ Successfully saved 50 samples to /content/data/baseline_eval


### 4. Run SQA Benchmark
Perform the `run_sqa_eval` passing the model directly locally. Metrics push to weights & biases.

In [5]:
# Cell 4: Run SQA Benchmark — Full Metrics Suite
import torch, json, os, sys
sys.path.insert(0, '/content/moshilite/src')

from moshi.models.loaders import CheckpointInfo
from moshilite.eval.sqa import SQAEvaluator

BENCHMARK_DIR = "/content/data/baseline_eval"
OUTPUT_DIR = "/content/drive/MyDrive/moshilite/eval/baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load both models
print("Loading Moshi LM + Mimi codec...")
device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt = CheckpointInfo.from_hf_repo(
    hf_repo="kyutai/moshiko-pytorch-bf16"
)
moshi_lm = ckpt.get_moshi(device=device, dtype=torch.bfloat16)
mimi = ckpt.get_mimi(device=device)
print(f"✅ Moshi LM: {sum(p.numel() for p in moshi_lm.parameters())/1e9:.2f}B params")
print(f"✅ Mimi:     {sum(p.numel() for p in mimi.parameters())/1e6:.1f}M params")

# Create evaluator and run
evaluator = SQAEvaluator(model=moshi_lm, mimi=mimi, device=device)
metrics = evaluator.run_sqa_eval(BENCHMARK_DIR)

# Save full results (including per-sample transcriptions)
results_path = os.path.join(OUTPUT_DIR, "baseline_moshiko_sqa_metrics.json")
with open(results_path, 'w') as f:
    json.dump(metrics, f, indent=4)

# Save summary only (without per-sample data, for quick viewing)
summary = {k: v for k, v in metrics.items() if k != 'transcriptions'}
summary_path = os.path.join(OUTPUT_DIR, "baseline_summary.json")
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=4)

print(f"\n📁 Full results:  {results_path}")
print(f"📁 Summary:       {summary_path}")


Loading Moshi LM + Mimi codec...


/usr/local/lib/python3.12/dist-packages/moshi/models/loaders.py:204: UserWarning: Repository kyutai/moshiko-pytorch-bf16 contains no config.json. Assuming this is a Moshi 7B. Support for such repository might be removed in the future.
  warnings.warn(


✅ Moshi LM: 7.69B params
✅ Mimi:     79.3M params
Loading Whisper (base)...
  Model: 7.69B params (LM) + 79.3M params (Mimi)

  [1] librispeech_asr_0.wav
      RTF=0.282 | VRAM=16.5GB | Time=12.448s


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


    [WARN] UTMOSv2 init failed: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434
    [WARN] UTMOS computation failed: 'NoneType' object has no attribute 'predict'
      EM=0 | F1=0.0 | WER=1.0 | BERT=0.8178 | UTMOS=-1.0
      Expected:  CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
      Got:       Hello, what's out?

  [2] librispeech_asr_1.wav
      RTF=0.963 | VRAM=16.51GB | Time=14.766s


...

    [WARN] UTMOS computation failed: 'NoneType' object has no attribute 'predict'
      EM=0 | F1=0.0 | WER=1.0 | BERT=0.8206 | UTMOS=-1.0
      Expected:  ISN'T HE THE GREATEST FOR GETTING INTO ODD CORNERS
      Got:       Hello, how was your day?

  [49] librispeech_asr_48.wav
      RTF=0.889 | VRAM=16.51GB | Time=9.307s


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    [WARN] UTMOS computation failed: 'NoneType' object has no attribute 'predict'
      EM=0 | F1=0.0 | WER=1.0 | BERT=0.8162 | UTMOS=-1.0
      Expected:  FORGETTING ALL THEIR WEARINESS THEY SEIZED THEIR CANDLES AND SCURRIED THROUGH TH
      Got:       Hey, how is it going?

  [50] librispeech_asr_49.wav
      RTF=0.918 | VRAM=16.51GB | Time=10.759s


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    [WARN] UTMOS computation failed: 'NoneType' object has no attribute 'predict'
      EM=0 | F1=0.1111 | WER=1.0 | BERT=0.8175 | UTMOS=-1.0
      Expected:  WELL I'M CONVINCED THAT THE BOARDED UP HOUSE MYSTERY HAPPENED NOT EARLIER THAN A
      Got:       Hi, how are you doing? What are you talking about? You know the book, I'm saying

  RESULTS SUMMARY (50 samples)
  Semantic:
    Exact Match:  0.0
    Token F1:     0.0204
    WER:          1.0085
    BERTScore F1: 0.8188
  Audio Quality:
    UTMOS (1-5):  -1.0
  Efficiency:
    Avg RTF:      0.783
    Peak VRAM:    16.51 GB
    LM Params:    7.688B

📁 Full results:  /content/drive/MyDrive/moshilite/eval/baseline/baseline_moshiko_sqa_metrics.json
📁 Summary:       /content/drive/MyDrive/moshilite/eval/baseline/baseline_summary.json


COMPONENT LEVEL BENCHMARK

In [6]:
# Cell 5: Full Component-Level Evaluation (A + B + C + D)
import glob, json, os, sys
sys.path.insert(0, '/content/moshilite/src')

from moshilite.eval.component_eval import ComponentEvaluator

BENCHMARK_DIR = "/content/data/baseline_eval"
METADATA_PATH = os.path.join(BENCHMARK_DIR, "metadata.jsonl")
BASELINE_SAVE_DIR = "/content/drive/MyDrive/moshilite/eval/baseline/component_baseline"

# moshi_lm and mimi already loaded from Cell 4
comp_eval = ComponentEvaluator(moshi_lm=moshi_lm, mimi=mimi, device=device)

# Run all 4 components with semantic eval via metadata.jsonl
results = comp_eval.evaluate(
    audio_paths=[],             # Will be auto-loaded from metadata
    metadata_path=METADATA_PATH,  # Provides both audio paths + expected texts
    save_baseline_to=BASELINE_SAVE_DIR,
)

# Save results
results_path = os.path.join(BASELINE_SAVE_DIR, "component_results.json")
with open(results_path, 'w') as f:
    json.dump(results.to_dict(), f, indent=4, default=str)
print(f"\n📁 Full results: {results_path}")



  === Component A: Mimi Codec Roundtrip ===
    PESQ=1.795 | STOI=0.8937 | SNR=-0.68dB (50 samples, 17.0s)

  === Components B+C: Temporal & Depth Transformer ===
    Text token agreement vs baseline: -1.0
    Hidden cosine sim vs baseline:    -1.0
    Per-CB accuracy vs baseline:      [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
    Mean CB accuracy:                 -1.0
    (50 samples, 404.5s)

  === Component D: Full Pipeline Semantic Correctness ===


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


    [WARN] UTMOSv2 init failed: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434
    [1] EM=0 F1=0.000 WER=1.000 BERT=0.833 UTMOS=-1.00 RTF=0.71
        Exp: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
        Got: Hi, how is it going?


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    [2] EM=0 F1=0.211 WER=0.930 BERT=0.823 UTMOS=-1.00 RTF=0.97
        Exp: THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY H
        Got: I was a killing. What is diverg in the sentence?


...

    [48] EM=0 F1=0.000 WER=1.000 BERT=0.820 UTMOS=-1.00 RTF=0.67
        Exp: ISN'T HE THE GREATEST FOR GETTING INTO ODD CORNERS
        Got: Hi, how's your day?


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    [49] EM=0 F1=0.000 WER=1.000 BERT=0.814 UTMOS=-1.00 RTF=0.89
        Exp: FORGETTING ALL THEIR WEARINESS THEY SEIZED THEIR CANDLES AND SCURRIED 
        Got: Hello? How is it going?


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    [50] EM=0 F1=0.000 WER=1.000 BERT=0.818 UTMOS=-1.00 RTF=0.91
        Exp: WELL I'M CONVINCED THAT THE BOARDED UP HOUSE MYSTERY HAPPENED NOT EARL
        Got: Hi, how is it going?
    EM=0.0 | F1=0.0168 | WER=1.0045 | BERT=0.8189 | UTMOS=-1.0 | RTF=0.789
    (50 samples, 510.5s)

  ✅ Baseline saved to /content/drive/MyDrive/moshilite/eval/baseline/component_baseline

  COMPONENT EVALUATION SUMMARY
  A) Mimi Codec:   PESQ=1.795 | STOI=0.8937 | SNR=-0.68dB
  B) Temporal LM:  TextAgree=-1.0 | HiddenCos=-1.0
  C) DepthTxfmr:   CB_Acc=[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
  D) Semantic:     EM=0.0 | F1=0.0168 | WER=1.0045 | BERT=0.8189 | UTMOS=-1.0
  Efficiency:      7.688B params | VRAM=16.78GB | RTF=0.789

📁 Full results: /content/drive/MyDrive/moshilite/eval/baseline/component_baseline/component_results.json
